# SymPy Interoperability

## Prompt

In addition to the types of algegras that can be generated by functions in 'algebra_generators.py'. Can you suggest other ways to generate groups or other algebraic structures represented in 'finalg'?

## Answer

This is directly aligned with your Cayley-Dickson interests. Let me also check `permutation.py`, `abstract_matrix.py`, and the JSON/examples mechanism briefly for a complete picture.I have a full picture now. Let me check the `data/algebras` JSON directory and `Examples` class briefly too.Beyond `algebra_generators.py`, your codebase already has several other construction mechanisms scattered across other modules — some are ready to use as-is, one lines up nicely with your own Cayley-Dickson interests. Here's what's actually there:

**Combine algebras you already have**
- `alg1 * alg2` (direct product, in `magma.py`) and `alg ** n` for repeated self-products.
- `alg.proper_subalgebras()` / `alg.subalgebra_from_elements(elements)` — pull subgroups/subrings out of something bigger.
- `group.quotient_group(normal_subgroup)` (or `group / normal_subgroup`) — build a smaller group from cosets.
- `group.commutator_subalgebra()`, `monoid.units_subgroup()`, `magma.center_algebra()` — derived substructures with specific algebraic meaning.
- `monoid.regular_representation()` — gives you a matrix-based algebra isomorphic to the original, if you want a concrete linear representation.

**Cayley-Dickson construction (`ring.py`)** — `ring.make_cayley_dickson_algebra(mu=None, version=1..4)` builds a new Ring/Field of twice the dimension from an existing one, with four different multiplication conventions (Schafer 1954/1966, Baez 2001, or none). There's also the older `ring.sqr()`. Given your work on `gint`'s Gaussian integers and the `G_n = F_n[x]/(x²+1)` conjecture, this is probably the most directly relevant one here — applying it to `generate_algebra_mod_n(p)` output is exactly how you'd get finite analogs of the Gaussian integers/quaternions inside `finalg` itself, and there's a `docs/55_cayley_dickson.rst` write-up plus an `is_gaussian_prime` helper already built around that use case.

**Matrix rings** — `generate_nxn_matrix_algebra(ring)` in `algebra_generators.py` builds the ring of 2×2 `AbstractMatrix` objects over any existing Ring/Field (currently hardcoded to n=2, per its TODO).

**The general-purpose one: `generate_algebra_from_element_dict`** — this is the most flexible tool in the repo and probably underused. You give it: a dict of named generator *objects* (they don't have to be strings — permutations, matrices, integers, tuples, anything), a binary operation on those objects, and (optionally) an equality test and immutability coercion. It computes the closure under that operation automatically and hands back a `finalg` algebra. This is literally how `generate_dihedral_group` is built, using your `Perm` class. It means you can generate:
- Any permutation group as a subgroup of $S_n$, just by naming a few `Perm` generators.
- Matrix groups (e.g. $GL_n$ or $SL_n$ over a finite ring) using `AbstractMatrix` as the generator objects.
- Groups defined by *any* Python class with `__mul__`/`__eq__` you write yourself — modular exponentiation, custom transformations, whatever.

**Hand-built or imported Cayley tables** — `make_finite_algebra(name, desc, elements, table)` accepts a raw table directly, and also accepts a JSON file path or dict (`src/finalg/data/algebras/*.json` shows the schema — just `name`/`description`/`elements`/`table`, plus `table2` for rings). That means you can transcribe multiplication tables from external sources — GroupNames.org, GAP's Small Groups Library, a textbook — into JSON and load them straight in, which is presumably how algebras like `Pinter29` or the Kandasamy example in your `data/algebras/` directory got there.

If it'd help, I could sketch a converter that pulls a group definition from SymPy's `combinatorics.PermutationGroup` (which already knows a lot of named groups) into `finalg`'s format via `generate_algebra_from_element_dict` — that'd give you a fast path to a much wider library of groups without hand-authoring JSON. Want me to put that together?

## Prompt

Yes, write a converter that pulls a group definition from SymPy into finalg's format.

## Answer

Good, sizes match here. Let's test a trickier group where a generator might not move the last points (size mismatch risk), plus test multiplication/equality semantics.Now let's validate it thoroughly, including cross-checking against finalg's own generators using `fast_isomorphic`.Now let's test a custom, arbitrary permutation group and a group derived from SymPy's own group-theory tools (derived subgroup, Sylow subgroup), plus test error handling.All working correctly, including the full symmetric group S5 (order 120) generated from just two permutations. Let's also add unit tests and run the full test suite, then check `elem_dict` round-trip use, and export the new function from `finalg/__init__.py`.sympy is already a declared dependency — good, no need for lazy imports. Now let's wire it into `__init__.py` and add tests.Now let's write unit tests, matching repo conventions (looking at `test_algebra_generators.py` for style, since that's the closest analog).

All pass, though 36s is a bit slow (mostly SymmetricGroup(4)/order-24 closure). Let's run the full suite to make sure nothing else broke.The 31-second time for S5 is too slow for what should be routine — that's from finalg's generic closure algorithm re-deriving something SymPy already knows how to compute efficiently. Let me switch to using SymPy's own element enumeration directly, which is dramatically faster.SymPy enumerates S5's 120 elements in 0.3ms versus 31s for the naive closure. Let's rewrite the converter to build the table directly from SymPy's enumeration.Now let's re-run all the validation checks and benchmark the new implementation.Found it — the bottleneck is SymPy's `Permutation.__mul__` object overhead, not the algorithm. Since `(p*q)[i] = q[p[i]]`, I can do the O(n²) table fill with plain integer tuples instead of Permutation objects, and only use SymPy for enumeration.Let me check what's actually slow — the table build itself, or something downstream in `make_finite_algebra`.That confirms it: `make_finite_algebra`'s eager `is_associative()` check is a pure-Python O(n³) triple loop — the real bottleneck, not my table construction. Since a `PermutationGroup` from SymPy is *mathematically guaranteed* to be associative with inverses, I can bypass that redundant check by constructing the `Group` directly.Now let's also update the module-level comment at the top (still references the old approach) and re-test everything.Now let's re-run the full benchmark and correctness suite.Excellent — S6 (order 720) now converts in 0.28s instead of 190s+. Let's re-run the `verify=True` path and all correctness checks against `fast_isomorphic`, then the full test suite.All correctness checks pass with the explicit `verify=True` re-check. Now run the full test suite again.Let's add tests for the `verify` parameter and a larger-group smoke test, then finalize.From 36.6s to 1.4s for the whole test file. Let's run the complete suite one final time to confirm everything's solid.All 549 tests pass in 14.7s total. Now let's package the final diff and files.Let's generate a patch containing only the new work (not re-including the already-delivered `fast_isomorphic` changes), plus copy the full new files for reference.Clean. Let's copy everything to outputs and present.Done — and the extra time was worth it. The first working version was correct but had a real performance problem, so I dug in and fixed it. Here's the story:

**What it does:** `finalg.sympy_interop.from_sympy_permutation_group(sympy_group, name=None, description=None, verify=False)` takes any `sympy.combinatorics.PermutationGroup` — `SymmetricGroup(n)`, `AlternatingGroup(n)`, `DihedralGroup(n)`, a custom group built from a couple of `Permutation` generators, or something SymPy derives (`.derived_subgroup()`, `.sylow_subgroup(p)`, etc.) — and returns `(finalg_group, elem_dict)`, where `elem_dict` maps each finalg element name back to its original SymPy `Permutation`.

**What changed along the way:**
- **v1** used `generate_algebra_from_element_dict` (the tool underlying your `generate_dihedral_group`), feeding it just the group's generators and letting finalg's own closure algorithm discover the rest. Correct, but S5 (order 120) took 31 seconds — the generic closure loop doesn't know anything about the group structure SymPy already has.
- **v2** enumerated elements via SymPy's own `PermutationGroup.generate()` (which uses proper group-theoretic algorithms) instead, and just filled in the table directly. Much better, but S6 (order 720) still took over 3 minutes.
- The actual culprit turned out to be one line inside `make_finite_algebra` itself: it always runs a pure-Python O(n³) triple loop to verify associativity before deciding what class to build. That's necessary in general (an arbitrary table might not be associative), but a `PermutationGroup` is associative and fully invertible *by definition* — so **v3** builds the `Group` object directly, skipping that redundant check, with an opt-in `verify=True` if you ever want the paranoid version back.

Net result: S6 (720 elements) went from "didn't finish" → 190s → **0.28s**.

I validated it by round-tripping through your own `fast_isomorphic` — `SymmetricGroup(3)`, `DihedralGroup(4)`, and `CyclicGroup(6)` from SymPy all check out isomorphic to your own `generate_symmetric_group(3)`, `generate_dihedral_group(4)`, and `generate_cyclic_group(6)` — plus tested a custom 2-generator subgroup that turns out to be all of S5, a derived subgroup (S4's derived subgroup correctly comes back as order 12, i.e. A4), and a Sylow-2 subgroup. All 16 new tests plus your full existing suite (549 tests total) pass.

The patch applies the same way as last time — from your repo root, `git apply ~/Downloads/sympy_converter.patch` (after `git apply --check` to confirm it's clean) — and it only touches `__init__.py` plus adds the two new files, so it won't conflict with the `fast_isomorphic` patch you already applied.

## EXAMPLE

In [3]:
from sympy.combinatorics import (
    Permutation,
    PermutationGroup,
    SymmetricGroup,
    AlternatingGroup,
    CyclicGroup,
    DihedralGroup,
)

from finalg.sympy_interop import from_sympy_permutation_group, to_sympy_permutation_group
from finalg.group import Group
from finalg import (
    generate_symmetric_group,
    generate_dihedral_group,
    generate_cyclic_group,
    make_finite_algebra,
    examples,
)

In [4]:
z4sp = CyclicGroup(4)
z4sp

PermutationGroup([
    (0 1 2 3)])

In [5]:
z4_from_sp, mapping = from_sympy_permutation_group(CyclicGroup(4), name='MyZ4', description='custom desc')

In [6]:
z4_from_sp.about()


** Group **
Name: MyZ4
Instance ID: 5054311472
Description: custom desc
Order: 4
Identity: '(0)(3)'
Commutative? Yes
Cyclic?: Yes
Generators: ['(0 1 2 3)', '(0 3 2 1)']
Elements:
   Index   Name   Inverse  Order
      0 '(0)(3)' '(0)(3)'       1
      1 '(0 1 2 3)' '(0 3 2 1)'       4
      2 '(0 2)(1 3)' '(0 2)(1 3)'       2
      3 '(0 3 2 1)' '(0 1 2 3)'       4
Cayley Table (showing indices):
[[0, 1, 2, 3], [1, 2, 3, 0], [2, 3, 0, 1], [3, 0, 1, 2]]


'<Group:MyZ4, ID:5054311472>'

In [7]:
mapping

{'(0)(3)': Permutation(3),
 '(0 1 2 3)': Permutation(0, 1, 2, 3),
 '(0 2)(1 3)': Permutation(0, 2)(1, 3),
 '(0 3 2 1)': Permutation(0, 3, 2, 1)}

# From finalg to SymPy

## Prompt

Does it make sense to try to create a method that does the reverse? That is, convert a 'finalg' algebra into a SymPy algebra?

## Answer

Good question — this deserves a careful answer, not just "sure, sounds symmetric." Let me check what's actually well-defined here before building anything.Yes — but only for **Groups**, and it took a bit of care to get right. Here's why, then the implementation.

**Why it makes sense for Groups:** Cayley's theorem says every finite group of order *n* embeds faithfully into the symmetric group $S_n$ via its regular representation — map each element $g$ to the permutation "multiply by $g$." That's exactly the reverse of what `from_sympy_permutation_group` does, and it's always well-defined for a `Group`, since a group's operation is guaranteed associative with inverses.

One subtlety I had to check empirically rather than assume: SymPy composes permutations as $(p \cdot q)(i) = q(p(i))$, so the map has to be *right* multiplication, $R_g(x) = x \cdot g$, not left — I verified this on `S3` above; left multiplication turns out to be an anti-homomorphism under SymPy's convention (it'd silently give you the *opposite* group's table, not a bug you'd necessarily notice until products came out transposed).

**Why it doesn't extend to your other algebra types:**
- **Quasigroup/Loop** — cancellative, but *not associative*. The regular-representation map is only a homomorphism *because* of associativity (that's the whole proof of Cayley's theorem); without it, composing two SymPy permutations wouldn't correspond to the loop's product at all. Wrapping that in a `PermutationGroup` wouldn't just lose information, it'd actively misrepresent the structure — the group SymPy computes by closing those permutations under composition would generally have nothing to do with your loop.
- **Magma/Semigroup without cancellation** — "multiply by $g$" isn't even guaranteed to be a bijection (e.g. an idempotent element can collapse points), so it isn't a *permutation* at all, just a general function. SymPy's combinatorics module doesn't have a "finite transformation semigroup" class to target.
- **Ring/Field** — two operations; SymPy's combinatorics module models single-operation permutation groups. You could embed the additive group alone, but that throws away multiplication and would be a partial, misleading "conversion" rather than a faithful one.

So the honest scope is: **Group → SymPy PermutationGroup**, via the right-regular representation. Let me build that.Now let's add the reverse direction to the same module.Now let's append the reverse converter:Now let's update the module-level header comment to mention both directions, then test thoroughly.Now let's install and test thoroughly, including checking exact table fidelity (not just isomorphism) and round-tripping.All exact homomorphisms confirmed — not just isomorphic, the SymPy multiplication table reproduces finalg's table exactly. Now let's test the round-trip, error handling, and larger groups.Now let's test with a Ring/Field (also should be rejected since `Ring`/`Field` subclass `Group` for the additive part — need to check this carefully).Good catch needed — `Field` and `Ring` are actually subclasses of `Group` in this hierarchy (they reuse `Group` for the additive part), so my `isinstance(Group)` check would silently let one through and only convert its *additive* structure, throwing away multiplication. That's exactly the misleading case I warned about — let me guard against it explicitly.Now add the `Ring` import:Now let's re-run the full validation suite and check performance for a larger group.`_smallest_generating_set()` — reused from `fast_isomorphic` — is the bottleneck here, not my new code. But I don't actually need a *minimal* generating set: SymPy's `PermutationGroup` works fine with redundant generators, and I've already computed permutations for every element anyway. Let me just skip that search.From 10s down to 0.01s. Let's update the implementation to drop the minimal-generating-set search entirely.Now let's also update the docstring line that referenced the generating-set approach:Now let's re-verify performance and correctness end-to-end.A5 went from 10s to 0.018s. Let's run the round-trip and error-handling tests once more to confirm nothing regressed, then run the full suite.All good. Now let's write unit tests for `to_sympy_permutation_group` in the test file.Now let's append a new test class for `to_sympy_permutation_group` at the end of the file.All 27 tests pass in 2.4s. Let's run the complete project test suite one final time.All 560 tests pass. Let's finalize and package the deliverables.Since the previous `sympy_converter.patch` was already applied, let me generate a clean incremental patch against exactly that prior state (using the files I delivered last time), so it applies without conflicts.Let's verify this incremental patch actually applies cleanly against a fresh copy that has only the previous patch applied (simulating the user's actual state).This confirms it — the user already applied and pushed the earlier patches, so the upstream repo now already has `fast_isomorphic` and `from_sympy_permutation_group`. Let me regenerate the incremental patch against a genuinely fresh clone of their current repo.Now let's verify it applies cleanly against the genuinely fresh clone, then run the full test suite to be sure.Confirmed working end-to-end on a real, clean simulation of the user's actual repo state. Let's also try `git apply` (in case they prefer it over `patch`) to make sure it works both ways.Works with both `patch` and `git apply` (just a harmless trailing-whitespace warning). Let's copy everything to outputs and present.So yes, it makes sense — but only for `Group`, and it's now in `finalg.sympy_interop` alongside the forward direction. Summary of what I found and built:

**`to_sympy_permutation_group(finalg_group, verify_order=True)`** embeds a finalg `Group` into SymPy via its regular representation (Cayley's theorem). Two things worth flagging that weren't obvious going in:

1. **Composition order matters.** SymPy composes as $(p \cdot q)(i) = q(p(i))$. I tested both left- and right-regular representations empirically on `S3` — left multiplication is an *anti*-homomorphism under that convention (it'd silently give you the opposite group), while right multiplication ($R_g(x) = x \cdot g$) reproduces your table exactly. I verified this isn't just "isomorphic" but a genuine exact match — `R_a * R_b == R_{a·b}` for every pair, checked on `S3`, `D4`, `Z6`, and the non-abelian `Q8`.

2. **`Ring`/`Field` needed an explicit guard.** They're actually subclasses of `Group` in your hierarchy (reusing it for the additive structure), so a naive `isinstance(Group)` check would silently let one through and convert *only* addition, quietly discarding multiplication. I added an explicit check that rejects them with an explanation and points to `units_subgroup()` if what they actually want is the multiplicative group.

**Where I had to walk back an initial mistake:** the first version reused the `_smallest_generating_set()` helper from `fast_isomorphic` — reasonable-looking symmetry with the forward direction, but that helper does its own combinatorial search for a *minimal* generating set, which turned out to dominate runtime (10s for `A5`, order 60). SymPy doesn't need a minimal set at all — Schreier-Sims handles redundant generators fine, and I'd already computed the permutation for every element anyway — so I just hand SymPy the whole list. That dropped A5 to 0.02s.

**What's rejected, and why:** `Magma`/`Quasigroup`/`Loop` (no associativity, so the regular-representation map isn't a homomorphism — composing the resulting permutations wouldn't reproduce the original product at all) and `Ring`/`Field` (two operations, as above) all raise `TypeError` with an explanation.

27 new tests (round-tripping through `fast_isomorphic`, exact-homomorphism checks, rejection cases, an order-60 performance guard), full suite at 560 tests passing.

Since the earlier patches are already live on your GitHub repo, `to_sympy_converter.patch` is scoped as a clean diff against *that* state — verified against a fresh clone of your actual upstream. Apply the same way as before: `git apply --check to_sympy_converter.patch` then `git apply to_sympy_converter.patch` from the repo root.

## EXAMPLE

In [8]:
z4 = generate_cyclic_group(4)
z4

Group(
'Z4',
'Autogenerated cyclic group of order 4',
('0', '1', '2', '3'),
[[0, 1, 2, 3], [1, 2, 3, 0], [2, 3, 0, 1], [3, 0, 1, 2]]
)

In [11]:
sympy_group, _ = to_sympy_permutation_group(z4)

In [12]:
sympy_group

PermutationGroup([
    (0 1 2 3),
    (0 2)(1 3),
    (0 3 2 1)])